# Exercise 1 - Sentiment Analysis: dataset, tokenizer e baseline stabile

Kernel usato: `DLA2026-transformers`.

Obiettivo: costruire una baseline semplice ma solida per sentiment analysis su Rotten Tomatoes partendo da DistilBERT pre-addestrato.

Il notebook copre tre passaggi incrementali:

1. caricamento ed esplorazione degli split del dataset;
2. caricamento di tokenizer e modello DistilBERT e ispezione degli output;
3. estrazione delle feature `[CLS]` e training di una SVM lineare come baseline stabile.

Disclosure AI: la struttura del notebook e' stata riorganizzata con supporto di ChatGPT/Codex. Il codice e' stato separato in funzioni riusabili; risultati e conclusioni sono stati controllati sulla run corrente.

> **Execution note**
>
> Gli output visibili sono stati prodotti durante le esecuzioni finali o di validazione del laboratorio. Nella versione di consegna i training costosi sono disattivati di default quando sono controllati da flag; checkpoint e artefatti salvati vengono usati per consultazione rapida.

## Setup

Le funzioni riusabili sono nel modulo `src/dla_lab2`. Il notebook resta leggibile e mostra solo i passaggi sperimentali principali.

Nota tecnica: l'ambiente usa Python 3.14. Per evitare un bug di compatibilita' tra `datasets`, `dill` e il fingerprinting della cache, `load_rotten_tomatoes()` applica una piccola patch locale prima di caricare il dataset.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT

WindowsPath('DLA_2')

In [2]:
from dataclasses import asdict
import importlib

import pandas as pd
import torch

import dla_lab2.sentiment as sentiment
from dla_lab2.seed import set_seed

# Ricarichiamo il modulo per essere sicuri che Jupyter usi l'ultima versione
# delle funzioni definite in src/dla_lab2/sentiment.py.
importlib.reload(sentiment)

from dla_lab2.sentiment import (
    dataset_overview,
    evaluate_sklearn_classifier,
    extract_cls_features_with_pipeline,
    inspect_tokenizer_output,
    load_distilbert_base,
    load_rotten_tomatoes,
    sample_examples,
    train_linear_svm,
)

# Seed fisso: rende ripetibile la selezione degli esempi e la SVM.
set_seed(42)

# La run corrente ha usato CUDA. Se CUDA non fosse disponibile, il codice passa a CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
pipeline_device = 0 if torch.cuda.is_available() else -1
device

# I download del modello e l'estrazione completa sono opt-in nella consegna.
RUN_MODEL_INSPECTION = False
RUN_FEATURE_BASELINE = False


'cuda'

## Exercise 1.1 - Loading the Dataset Splits

Richieste dell'esercizio:

- caricare il dataset Cornell Rotten Tomatoes;
- capire quali split sono disponibili;
- esplorare organizzazione, colonne e label.

La tabella sotto mostra che il dataset e' gia' diviso in `train`, `validation` e `test`; ogni split contiene le colonne `text` e `label`, con label binarie `[0, 1]`.

In [3]:
# Carichiamo il dataset Rotten Tomatoes e riassumiamo gli split disponibili.
# L'output osservato nella run e': train=8530, validation=1066, test=1066.
ds_dict = load_rotten_tomatoes()
pd.DataFrame(dataset_overview(ds_dict))

<local-conda-env>\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,split,num_rows,columns,labels
0,train,8530,"[text, label]","[0, 1]"
1,validation,1066,"[text, label]","[0, 1]"
2,test,1066,"[text, label]","[0, 1]"


In [4]:
# Esploriamo alcuni esempi casuali del train split.
# Questo serve a vedere concretamente il formato: una frase breve e una label binaria.
pd.DataFrame(sample_examples(ds_dict["train"], n=8, seed=42))

,idx,label,text
0,733,1,a bold and subversive film that cuts across th...
1,5948,0,"if you believe any of this , i can make you a ..."
2,7322,0,the stories here suffer from the chosen format .
3,5580,0,"if religious films aren't your bailiwick , sta..."
4,3692,1,some remarkable achival film about how shangha...
5,3741,1,if a big musical number like 'praise the lord ...
6,760,1,andersson creates a world that's at once surre...
7,6597,0,"a hideous , confusing spectacle , one that may..."


Osservazioni Exercise 1.1:

- Gli split osservati sono `train` con 8530 esempi, `validation` con 1066 e `test` con 1066.
- Le colonne sono `text` e `label`.
- La label `0` rappresenta sentiment negativo; la label `1` rappresenta sentiment positivo.
- Gli esempi sono frasi brevi gia' preprocessate, adatte a un primo laboratorio sui transformer.
- Il warning su `IProgress`/`tqdm` non influisce sui dati: segnala solo che la barra di progresso interattiva di Jupyter non e' installata.

## Exercise 1.2 - A Pre-trained BERT and Tokenizer

Richieste dell'esercizio:

- caricare DistilBERT e il tokenizer corrispondente;
- tokenizzare alcuni esempi del dataset;
- passare i token nel modello;
- osservare quali output produce il modello.

Usiamo DistilBERT base senza testa di classificazione. In questa fase il modello non viene addestrato: lo usiamo solo per capire il formato delle rappresentazioni.

In [5]:
if RUN_MODEL_INSPECTION or RUN_FEATURE_BASELINE:
    # Carichiamo tokenizer e modello DistilBERT base.
    # Il report con pesi "UNEXPECTED" riguarda la testa language-modeling del checkpoint
    # e puo' essere ignorato qui: stiamo caricando il backbone DistilBERT senza quella testa.
    tokenizer, bert_model = load_distilbert_base(device=device)

    # Tokenizziamo tre frasi reali del train split.
    texts = [ds_dict["train"][i]["text"] for i in range(3)]
    batch = inspect_tokenizer_output(tokenizer, texts)

    # Verifichiamo le chiavi restituite dal tokenizer.
    batch.keys()
else:
    tokenizer = bert_model = batch = None
    print("Caricamento DistilBERT non eseguito in modalita' rapida.")


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3203.94it/s]
DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeysView({'input_ids': tensor([[  101,  1996,  2600,  2003, 16036,  2000,  2022,  1996,  7398,  2301,
          1005,  1055,  2047,  1000, 16608,  1000,  1998,  2008,  2002,  1005,
          1055,  2183,  2000,  2191,  1037, 17624,  2130,  3618,  2084,  7779,
         29058,  8625, 13327,  1010,  3744,  1011, 18856, 19513,  3158,  5477,
          4168,  2030,  7112, 16562,  2140,  1012,   102,     0,     0,     0,
             0,     0],
        [  101,  1996,  9882,  2135,  9603, 13633,  1997,  1000,  1996,  2935,
          1997,  1996,  7635,  1000, 11544,  2003,  2061,  4121,  2008,  1037,
          5930,  1997,  2616,  3685, 23613,  6235,  2522,  1011,  3213,  1013,
          2472,  2848,  4027,  1005,  1055,  4423,  4432,  1997,  1046,  1012,
          1054,  1012,  1054,  1012, 23602,  1005,  1055,  2690,  1011,  3011,
          1012,   102],
        [  101,  4621,  2021,  2205,  1011,  8915, 23267, 16012, 24330,   102,
             0,     0,     0,     0,     0,     0,     0,   

In [6]:
if RUN_MODEL_INSPECTION:
    # Decodifichiamo la prima sequenza per vedere i token speciali.
    # [CLS] apre la sequenza, [SEP] la chiude, [PAD] riempie fino alla lunghezza massima del batch.
    print(tokenizer.decode(batch["input_ids"][0]))

    # Nella run osservata il batch ha 3 frasi, tutte portate a lunghezza 52.
    print({key: value.shape for key, value in batch.items()})
else:
    print("Ispezione tokenizer non eseguita; l'output finale resta visibile nel notebook.")


[CLS] the rock is destined to be the 21st century ' s new " conan " and that he ' s going to make a splash even greater than arnold schwarzenegger, jean - claud van damme or steven segal. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD]
{'input_ids': torch.Size([3, 52]), 'token_type_ids': torch.Size([3, 52]), 'attention_mask': torch.Size([3, 52])}


In [7]:
if RUN_MODEL_INSPECTION:
    # Passiamo il batch nel modello senza gradienti: e' solo un'ispezione degli output.
    with torch.no_grad():
        outputs = bert_model(**{key: value.to(device) for key, value in batch.items()})

    # Output osservato: torch.Size([3, 52, 768]).
    # Significa: 3 frasi, 52 token per frase, vettore da 768 dimensioni per ogni token.
    outputs.last_hidden_state.shape
else:
    print("Forward pass non eseguito in modalita' rapida.")


torch.Size([3, 52, 768])

Interpretazione Exercise 1.2:

- `input_ids` contiene gli indici numerici dei token.
- `attention_mask` vale 1 sui token reali e 0 sui padding.
- `token_type_ids` compare nel batch, ma per DistilBERT non e' la parte concettualmente importante in questo esercizio.
- `last_hidden_state` ha forma `(batch, sequence_length, hidden_size)`; nella run e' `(3, 52, 768)`.
- Per la baseline dell'esercizio 1.3 usiamo il vettore del primo token, cioe' `[CLS]`, come rappresentazione sintetica della frase.

## Exercise 1.3 - Stable Baseline

Richieste dell'esercizio:

1. usare DistilBERT come feature extractor;
2. addestrare un classificatore a scelta sulle feature estratte;
3. valutare performance su validation e test.

Qui DistilBERT resta congelato. La funzione `extract_cls_features_with_pipeline()` usa la pipeline Hugging Face `feature-extraction`, interpreta l'output e conserva solo il token `[CLS]` dell'ultimo layer. Sopra queste feature addestriamo una SVM lineare di Scikit-learn.

In [8]:
if RUN_FEATURE_BASELINE:
    features = {}
    for split in ["train", "validation", "test"]:
        # Convertiamo i testi dello split in lista per passarli alla pipeline Hugging Face.
        texts = list(ds_dict[split]["text"])

        # Estraiamo una matrice di feature: una riga per frase, 768 colonne per DistilBERT base.
        features[split] = extract_cls_features_with_pipeline(
            texts=texts,
            model=bert_model,
            tokenizer=tokenizer,
            batch_size=32,
            device=pipeline_device,
        )

        # Shape osservate nella run: train (8530, 768), validation (1066, 768), test (1066, 768).
        print(split, features[split].shape)
else:
    features = {}
    print("Feature extraction non eseguita in modalita' rapida.")


train (8530, 768)
validation (1066, 768)
test (1066, 768)


In [9]:
if RUN_FEATURE_BASELINE:
    # Addestriamo la SVM solo sulle feature del train split.
    svm = train_linear_svm(features["train"], ds_dict["train"]["label"])

    # Valutiamo su validation e test, cioe' su dati non usati per addestrare la SVM.
    baseline_metrics = [
        asdict(evaluate_sklearn_classifier(svm, features["validation"], ds_dict["validation"]["label"], "validation")),
        asdict(evaluate_sklearn_classifier(svm, features["test"], ds_dict["test"]["label"], "test")),
    ]
    pd.DataFrame(baseline_metrics)
else:
    display(pd.read_csv(PROJECT_ROOT / "results" / "sentiment_results.csv").query("method == 'DistilBERT CLS + linear SVM'"))


,split,accuracy,f1,precision,recall
0,validation,0.818011,0.814176,0.831703,0.797373
1,test,0.794559,0.790831,0.805447,0.776735


## Conclusioni Exercise 1

Tutti i punti richiesti sono stati svolti.

Exercise 1.1:
- dataset caricato correttamente;
- split disponibili identificati: `train`, `validation`, `test`;
- struttura verificata: colonne `text` e `label`, label binarie `[0, 1]`.

Exercise 1.2:
- tokenizer e modello DistilBERT caricati;
- esempi reali tokenizzati;
- output del modello ispezionato: `last_hidden_state` con forma osservata `(3, 52, 768)`.

Exercise 1.3:
- DistilBERT usato come feature extractor congelato;
- feature `[CLS]` estratte per tutti gli split con shape coerenti;
- SVM lineare addestrata sul train split;
- performance valutata su validation e test.

Risultati osservati della baseline:

- validation accuracy: `0.8180`, F1: `0.8142`;
- test accuracy: `0.7946`, F1: `0.7908`.

Interpretazione: la baseline e' coerente e credibile. La performance sul test e' leggermente piu' bassa della validation, ma il calo e' contenuto. Questo sara' il riferimento da confrontare con il fine-tuning completo di DistilBERT nel notebook successivo.

## Referenced functions and source files

| Function/class | Defined in | Purpose |
| --- | --- | --- |
| `load_rotten_tomatoes` | `src/dla_lab2/sentiment.py` | Caricamento dataset sentiment. |
| `extract_cls_features_with_pipeline` | `src/dla_lab2/sentiment.py` | Feature extraction da DistilBERT. |
| `build_trainer` | `src/dla_lab2/sentiment.py` | Configurazione Hugging Face Trainer. |
| `CLIPAdapter` | `src/dla_lab2/clip_utils.py` | Adattatore leggero per feature CLIP. |
